In [12]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score

Check Device (CPU/GPU)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


Define the GCN Model (CORE MODEL)

In [14]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x

Training Function (Reusable for Any Hospital)

In [20]:
def train_local_model(graph_path, epochs=100, lr=0.01):
    # Load graph
    graph = torch.load(graph_path, weights_only=False)
    graph = graph.to(device)

    # Initialize model
    model = GCN(
        input_dim=graph.num_node_features,
        hidden_dim=32,
        output_dim=2
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    # Training loop
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        out = model(graph.x, graph.edge_index)
        loss = criterion(out, graph.y)

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"[{graph_path}] Epoch {epoch:03d} | Loss: {loss:.4f}")

    return model

Train Models at Each Hospital

In [21]:
model_A = train_local_model("../data/graph/graph_A.pt")
model_B = train_local_model("../data/graph/graph_B.pt")
model_C = train_local_model("../data/graph/graph_C.pt")

[../data/graph/graph_A.pt] Epoch 000 | Loss: 0.5807
[../data/graph/graph_A.pt] Epoch 020 | Loss: 0.0570
[../data/graph/graph_A.pt] Epoch 040 | Loss: 0.0446
[../data/graph/graph_A.pt] Epoch 060 | Loss: 0.0382
[../data/graph/graph_A.pt] Epoch 080 | Loss: 0.0316
[../data/graph/graph_B.pt] Epoch 000 | Loss: 0.6049
[../data/graph/graph_B.pt] Epoch 020 | Loss: 0.0950
[../data/graph/graph_B.pt] Epoch 040 | Loss: 0.0643
[../data/graph/graph_B.pt] Epoch 060 | Loss: 0.0678
[../data/graph/graph_B.pt] Epoch 080 | Loss: 0.0544
[../data/graph/graph_C.pt] Epoch 000 | Loss: 1.1589
[../data/graph/graph_C.pt] Epoch 020 | Loss: 0.0321
[../data/graph/graph_C.pt] Epoch 040 | Loss: 0.0226
[../data/graph/graph_C.pt] Epoch 060 | Loss: 0.0163
[../data/graph/graph_C.pt] Epoch 080 | Loss: 0.0160


In [23]:
def evaluate_model(model, graph_path):
    graph = torch.load(graph_path, weights_only=False).to(device)
    model.eval()

    with torch.no_grad():
        logits = model(graph.x, graph.edge_index)
        preds = logits.argmax(dim=1).cpu()
        labels = graph.y.cpu()

    acc = accuracy_score(labels, preds)
    return acc

print("Hospital A Accuracy:", evaluate_model(model_A, "../data/graph/graph_A.pt"))
print("Hospital B Accuracy:", evaluate_model(model_B, "../data/graph/graph_B.pt"))
print("Hospital C Accuracy:", evaluate_model(model_C, "../data/graph/graph_C.pt"))

Hospital A Accuracy: 0.990625
Hospital B Accuracy: 0.9829545454545454
Hospital C Accuracy: 1.0
